## 1. 管理工具安装
### 1.1 langchain 安装

In [1]:
!pip install langchain

# conda
# conda install -c conda-forge langchain

### 1.2 其他库安装

In [2]:
# 直接转换为 REST API
!pip install "langserve[all]"

# 观测平台
!pip install -U langsmith

# LangGraph
!pip install -U langgraph

import sys
sys.path.append('/Users/leimao/miniconda3/envs/milvus_faq/lib/python3.9/site-packages')

# DeepSeek
!pip install langchain-deepseek

# 外部资源集成
!pip install langchain_community

# dotenv管理密钥，加载环境变量等
!pip install python-dotenv

!pip list | grep langchain

langchain                               1.2.10
langchain-classic                       1.0.0
langchain-community                     0.4.1
langchain-core                          1.2.17
langchain-deepseek                      1.0.1
langchain-huggingface                   1.2.0
langchain-openai                        1.1.6
langchain-text-splitters                1.1.0


### 1.3 常见的处理链包含三个要素

1. 语言模型：核心推理引擎
2. 提示词模板：提供指令
3. 输出解释器：转换为易于使用的格式，便于下游处理

#### 1.3.1 语言模型

langchain语言模型主要分为两种：

1. LLM 通用模型：接收字符串作为输入，输出字符串作为输出。
2. ChatModel 对话模型：接收消息列表，输出“消息”，用于一问一答。

消息：由 BaseMessage 类定义，包含文本、时间戳、用户标识符等属性。主要的有

1. 消息的内容：文本，通常是字符串
2. 角色：消息的发送方

角色： LangChain 用于区分不同角色的对象

1. HumanMessage：人类（用户）输入的BaseMessage。
2. AIMessage：AI助手（大模型）输出的BaseMessage。
3. SystemMessage：系统预设的BaseMessage。
4. FunctionMessage：自定义函数输出的BaseMessage。
5. ToolMessage：调用第三方工具输出的BaseMessage。
6. ChatMessage：自定义角色。

方法： LCEL 默认实现同步调用方法，最常见的是 invoke 方法，接受一个 BaseMessage 对象作为参数，并返回一个 BaseMessage 对象作为结果。

比如：
LLMs.invoke：输入输出都是字符串
ChatModels.invoke：输入输出都是BaseMessage对象

In [3]:
from langchain_community.llms.tongyi import  Tongyi

llm = Tongyi()


print(llm.invoke("你好"))
print(llm("你好，今天天气怎么样？"))


你好！很高兴见到你～😊 有什么我可以帮你的吗？无论是学习、工作、生活中的问题，还是想聊聊天、获取信息、需要创意灵感，我都很乐意协助！


/var/folders/h6/fw8d9zmn0_b649z9ch8w4hy00000gn/T/ipykernel_52929/1056123211.py:7: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(llm("你好，今天天气怎么样？"))


你好！😊 不过我无法实时获取当前的天气信息，因为我没有联网功能，也无法访问你的设备定位或天气服务。

你可以试试以下方法快速查看今天天气：
- 在手机上打开自带的「天气」App（如苹果天气、华为天气、墨迹天气等）  
- 在微信中搜索“城市名+天气”，或使用“微信小程序”里的天气服务  
- 在浏览器搜索“北京天气”（把“北京”换成你所在的城市）  

如果你告诉我你所在的城市，我可以帮你分析当地常见的天气特点、穿衣建议，或者提供一份今日天气自查小贴士哦～ 🌤️☔🧥  
需要的话随时告诉我～


In [4]:
from langchain_deepseek import ChatDeepSeek

deepseek = ChatDeepSeek(model="deepseek-chat")

print(deepseek.invoke("你好"))

content='你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意为你提供支持！\n\n我可以帮你解答各种问题，协助处理文档，进行创作和分析等等。而且我是完全免费的哦！有什么我可以为你做的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 5, 'total_tokens': 75, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '4a585884-a562-4951-abd1-dd56f3ea05f5', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--ba1f0446-90ff-4448-a74f-bdd34ec4c16c-0' usage_metadata={'input_tokens': 5, 'output_tokens': 70, 'total_tokens': 75, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


#### 1.3.2 提示词模板

提示词模板是用于生成提示词的模板。例如，`{name}，你好！我是你的朋友小爱同学。`就是一个提示词模板。

提示词模板中可以使用变量，变量名由字母、数字和下划线组成，且必须以字母开头。例如，`{name}，你好！我是你的朋友小爱同学。`中的`{name}`就是一个变量。

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Hello, I am a {model_name}. How can I help you today?"
)
prompt.format(model_name="chatbot")


'Hello, I am a chatbot. How can I help you today?'

In [6]:
# 使用 ChatPromptTemplate 创建对话模板
from langchain_core.prompts import ChatPromptTemplate

# 创建包含 system 和 human 消息的聊天模板
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant named {assistant_name}."),
    ("human", "Hello, my name is {user_name}. {question}")
])

# 格式化消息
messages = chat_prompt.format_messages(
    assistant_name="Claude",
    user_name="Alice",
    question="What's the weather like today?"
)

for message in messages:
    print(f"{message.type}: {message.content}")


system: You are a helpful AI assistant named Claude.
human: Hello, my name is Alice. What's the weather like today?


In [7]:
from langchain_core.prompts import PromptTemplate

# 多变量模板
complex_prompt = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="""
    请为{audience}写一篇关于{topic}的文章。
    写作风格应该是{tone}的。

    文章要求：
    - 内容准确且有用
    - 结构清晰
    - 适合目标受众
    """
)

formatted_prompt = complex_prompt.format(
    topic="人工智能",
    audience="初学者",
    tone="通俗易懂"
)
print(formatted_prompt)



    请为初学者写一篇关于人工智能的文章。
    写作风格应该是通俗易懂的。

    文章要求：
    - 内容准确且有用
    - 结构清晰
    - 适合目标受众
    


In [8]:
from langchain_core.prompts import PromptTemplate

# f-string 风格、jinja2语法 ...
f_string_prompt = PromptTemplate.from_template(
    "分析以下{data_type}数据：\n{data}\n\n请提供{analysis_type}分析。"
)

result = f_string_prompt.format(
    data_type="销售",
    data="Q1销售额: 100万, Q2销售额: 120万",
    analysis_type="趋势"
)
print(result)


分析以下销售数据：
Q1销售额: 100万, Q2销售额: 120万

请提供趋势分析。


In [9]:
from langchain_core.prompts import PromptTemplate

# 带条件逻辑的模板
conditional_prompt = PromptTemplate(
    input_variables=["user_type", "question"],
    template="""
    {%- if user_type == "expert" -%}
    作为专家，请详细回答：{question}
    {%- else -%}
    请用简单易懂的方式回答：{question}
    {%- endif -%}
    """
)


#### 1.3.3 输出解析器

输出解析器是 LangChain 中的重要组件，用于将大语言模型(LLM)的原始文本输出转换为结构化数据格式，便于程序进一步处理和使用。

主要功能
- 格式化输出: 将 LLM 的自然语言输出解析为 JSON、XML、列表等结构化格式
- 类型转换: 将文本数据转换为 Python 对象（字典、列表、自定义类等）
- 数据验证: 确保输出符合预期的格式和约束条件
- 错误处理: 处理解析失败的情况，提供重试机制

常见的输出解析器类型



In [10]:
# JSON 输出解析器
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class PersonInfo(BaseModel):
    name: str = Field(description="人物姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

parser = PydanticOutputParser(pydantic_object=PersonInfo)

# 列表输出解析器

from langchain.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
# 将 "苹果,香蕉,橙子" 解析为 ["苹果", "香蕉", "橙子"]

# 结构化输出解析器
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

response_schemas = [
    ResponseSchema(name="answer", description="问题的答案"),
    ResponseSchema(name="source", description="答案来源")
]
parser = StructuredOutputParser.from_response_schemas(response_schemas)


In [11]:
# 手动使用解析器
parser = CommaSeparatedListOutputParser()

# 模拟 LLM 的原始输出
raw_output = "Python, Java, JavaScript, C++, Go"

# 解析输出
parsed_result = parser.parse(raw_output)
print("手动解析结果:", parsed_result)
# 输出: ['Python', 'Java', 'JavaScript', 'C++', 'Go']


手动解析结果: ['Python', 'Java', 'JavaScript', 'C++', 'Go']


In [12]:
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain_community.llms import Tongyi

# 初始化
parser = CommaSeparatedListOutputParser()
llm = Tongyi(temperature=0)

# 直接构建提示并调用
def simple_list_generation(category):
    # 手动构建提示
    prompt = f"""请列出5个{category}的例子。
{parser.get_format_instructions()}"""

    # 直接调用LLM
    response = llm.invoke(prompt)

    # 解析结果
    return parser.parse(response)


# 使用
fruits = simple_list_generation("水果")
print("水果列表:", fruits)

languages = simple_list_generation("编程语言")
print("编程语言列表:", languages)


水果列表: ['apple', 'banana', 'orange', 'grape', 'strawberry']
编程语言列表: ['Python', 'JavaScript', 'Java', 'C++', 'Ruby']


In [13]:
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.tongyi import Tongyi

# 1. 创建输出解析器
parser = CommaSeparatedListOutputParser()

# 2. 获取格式化指令
format_instructions = parser.get_format_instructions()
print("格式化指令:", format_instructions)

# 3. 创建包含格式指令的提示模板
prompt = PromptTemplate(
    template="请列出5个{category}的例子。\n{format_instructions}",
    input_variables=["category"],
    partial_variables={"format_instructions": format_instructions}
)

# 4. 初始化 LLM
llm = Tongyi(temperature=0)

# 5. 创建完整的链
chain = prompt | llm | parser

# 6. 使用示例
try:
    # 调用链并获取解析后的结果
    result = chain.invoke({"category": "水果"})
    print("解析后的结果:", result)
    print("结果类型:", type(result))
    # 输出示例: ['苹果', '香蕉', '橙子', '葡萄', '草莓']

    # 遍历结果
    for i, fruit in enumerate(result, 1):
        print(f"{i}. {fruit}")

except Exception as e:
    print(f"解析错误: {e}")

# 7. 更复杂的示例 - 多个类别
categories = ["编程语言", "运动项目", "乐器"]

for category in categories:
    print(f"\n=== {category} ===")
    try:
        result = chain.invoke({"category": category})
        for item in result:
            print(f"• {item}")
    except Exception as e:
        print(f"处理 {category} 时出错: {e}")


格式化指令: Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
解析后的结果: ['apple', 'banana', 'orange', 'grape', 'strawberry']
结果类型: <class 'list'>
1. apple
2. banana
3. orange
4. grape
5. strawberry

=== 编程语言 ===
• Python
• JavaScript
• Java
• C++
• Ruby

=== 运动项目 ===
• Running
• Swimming
• Cycling
• Basketball
• Tennis

=== 乐器 ===
• piano
• guitar
• violin
• flute
• drums


In [14]:
# 购物清单
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Tongyi

# 1. 初始化组件
parser = CommaSeparatedListOutputParser()
llm = Tongyi(temperature=0)

# 2. 创建提示模板
shopping_prompt = PromptTemplate(
    template="根据{meal_type}，生成一个包含5个食材的购物清单。\n{format_instructions}",
    input_variables=["meal_type"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 3. 创建链
shopping_chain = shopping_prompt | llm | parser

# 4. 生成不同餐型的购物清单
meals = ["早餐", "午餐", "晚餐"]
shopping_lists = {}

for meal in meals:
    shopping_lists[meal] = shopping_chain.invoke({"meal_type": meal})
    print(f"{meal}购物清单: {shopping_lists[meal]}")

# 5. 格式化输出
print("\n=== 详细购物清单 ===")
for meal, items in shopping_lists.items():
    print(f"\n🍽️ {meal}:")
    for i, item in enumerate(items, 1):
        print(f"  {i}. {item}")


早餐购物清单: ['eggs', 'whole grain bread', 'bananas', 'Greek yogurt', 'almonds']
午餐购物清单: ['chicken breast', 'brown rice', 'broccoli', 'cherry tomatoes', 'olive oil']
晚餐购物清单: ['chicken breast', 'broccoli', 'brown rice', 'garlic', 'soy sauce']

=== 详细购物清单 ===

🍽️ 早餐:
  1. eggs
  2. whole grain bread
  3. bananas
  4. Greek yogurt
  5. almonds

🍽️ 午餐:
  1. chicken breast
  2. brown rice
  3. broccoli
  4. cherry tomatoes
  5. olive oil

🍽️ 晚餐:
  1. chicken breast
  2. broccoli
  3. brown rice
  4. garlic
  5. soy sauce


In [15]:
# 多场景
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Tongyi

# 通用列表生成链
parser = CommaSeparatedListOutputParser()
list_chain = (
    PromptTemplate(
        template="请列出5个{category}的{item_type}。\n{format_instructions}",
        input_variables=["category", "item_type"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    | Tongyi(temperature=0)
    | parser
)

# 多场景使用
scenarios = [
    {"category": "早餐", "item_type": "食材"},
    {"category": "办公室", "item_type": "用品"},
    {"category": "旅行", "item_type": "必需品"},
    {"category": "健身", "item_type": "器材"},
    {"category": "学习", "item_type": "工具"}
]

for scenario in scenarios:
    result = list_chain.invoke(scenario)
    print(f"{scenario['category']}{scenario['item_type']}: {result}")


早餐食材: ['oatmeal', 'eggs', 'banana', 'whole-grain bread', 'almond milk']
办公室用品: ['pen', 'notebook', 'stapler', 'paperclip', 'printer']
旅行必需品: ['Passport', 'travel insurance', 'smartphone', 'charger', 'comfortable walking shoes']
健身器材: ['dumbbells', 'treadmill', 'resistance bands', 'pull-up bar', 'kettlebell']
学习工具: ['Anki', 'Notion', 'Coursera', 'Khan Academy', 'Obsidian']


#### 1.3.4 使用 LCEL 的好处

1. 简洁语法: 使用管道操作符 | 创建链，代码直观易读
2. 自动类型推断: 组件间数据类型自动转换，减少手动处理
3. 并行处理: 内置并行执行能力，提高处理效率
4. 流式处理: 原生支持流式输出，适合实时应用
5. 智能缓存: 自动缓存中间结果，避免重复计算
6. 内置调试: 更好的调试和监控能力
7. 插件化: 支持自定义组件和扩展
8. 标准接口: 统一的 Runnable 接口
9. 版本管理: 支持链的版本控制和管理